# 02 — The ResNet50 damage classifier

Goal: given an image, output a probability for each of the 6 damage types in `DAMAGE_TYPES`. This is a **multi-label** problem — one image can have both a dent *and* a scratch.

## Roadmap

1. What is a convolution? (with math + a tiny worked example)
2. From convolution → CNN → ResNet → ResNet50
3. The **residual** trick — why skip connections matter
4. Transfer learning: stage 1 (freeze) and stage 2 (fine-tune)
5. Loss function: binary cross-entropy for multi-label
6. **Runnable** demo-scale training cell (≤ 10 min on Colab)
7. **Optional** full-training cell (multi-hour)


In [ ]:
# === Colab bootstrap ===
# Safe to re-run. On a local clone with `pip install -e .` already done this
# is a no-op; on Colab it clones the repo + installs deps the first time.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/theDocWho/car-crash-fix-amount-predictor.git"
REPO_DIR = Path("car-crash-fix-amount-predictor")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

# Make `ccdp` importable whether or not the package was installed editable.
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("ccdp path:", src_path)
print("running in Colab:", IN_COLAB)


## 1. What is a convolution?

Imagine you have a 5×5 greyscale image and a 3×3 *kernel* (a tiny matrix of weights). You slide the kernel over the image, and at each position you compute one number:

$$
(I * K)(x, y) = \sum_{i=-1}^{1} \sum_{j=-1}^{1} I(x+i, y+j) \cdot K(i, j)
$$

That's it. Convolution is just "weighted sum of a neighborhood, repeated everywhere."

The magic: **the same kernel weights are used at every spatial position**. So a kernel that learns to detect "vertical edge" detects vertical edges anywhere in the image — top-left, bottom-right, anywhere. This is called *translation equivariance* and it is why CNNs work for images.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Vertical edge kernel — the classic "Sobel-x".
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)

# A test image: a sharp vertical edge in the middle.
img = np.zeros((9, 9), dtype=np.float32)
img[:, 4:] = 1.0

def conv2d(I, K):
    h, w = I.shape; kh, kw = K.shape
    out = np.zeros((h-kh+1, w-kw+1), dtype=np.float32)
    for y in range(out.shape[0]):
        for x in range(out.shape[1]):
            out[y, x] = (I[y:y+kh, x:x+kw] * K).sum()
    return out

response = conv2d(img, sobel_x)
print("response (3x3 sliding sum):"); print(response)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("input"); axes[0].axis("off")
axes[1].imshow(response, cmap="RdBu"); axes[1].set_title("Sobel-x response"); axes[1].axis("off")
plt.show()


Notice the response is **near zero** everywhere except *at the edge*, where it spikes. A trained CNN learns kernels like this — but for "is this part of a dent?" instead of "is this a vertical edge?"

## 2. From convolution → CNN → ResNet50

A CNN stacks many convolutions, each followed by:
- **Activation** (ReLU: $f(x) = \max(0, x)$ — keeps the network nonlinear)
- **Pooling** (downsample, e.g. max-pool 2×2 → halves spatial size)
- **Batch normalisation** (normalises layer outputs to mean 0, var 1; speeds training)

Early layers learn edges and color blobs. Middle layers learn textures and shapes. Late layers learn object parts.

**ResNet50** is a 50-layer CNN published by He et al. (2015). The "50" is the depth in weight layers.

## 3. The residual trick — why ResNet works at all

Naively stacking 50 layers does *not* work — training accuracy actually gets **worse** as you go deeper, because gradients vanish or explode during backprop.

ResNet's fix: in each block, instead of computing $y = F(x)$, compute

$$
y = F(x) + x
$$

The "+x" is the **skip connection** (or **residual** connection). The block now learns the *residual* $F(x) = y - x$ — the *change* you want to add to the input, not the whole output.

Why this helps:
- If the optimal $F$ is "do nothing", the network can drive $F(x) \to 0$ trivially. Without the skip, it would have to learn the identity function across all 50 layers — much harder.
- Gradients can flow straight back through the skip path, bypassing the deep layers, which keeps gradient magnitudes alive.


In [ ]:
# Visualise a residual block as a tiny diagram.
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlim(0, 10); ax.set_ylim(0, 4); ax.axis("off")
ax.add_patch(patches.FancyBboxPatch((0.5, 1.5), 1.2, 1, boxstyle="round,pad=0.05",
                                     facecolor="#bbdefb"))
ax.text(1.1, 2, "x", ha="center", va="center", fontsize=12)
for x, label, color in [(3, "conv\n3x3", "#c8e6c9"),
                         (5, "ReLU",     "#fff59d"),
                         (7, "conv\n3x3", "#c8e6c9")]:
    ax.add_patch(patches.FancyBboxPatch((x, 1.5), 1.2, 1,
                  boxstyle="round,pad=0.05", facecolor=color))
    ax.text(x+0.6, 2, label, ha="center", va="center", fontsize=10)
# main path arrows
for a, b in [((1.7,2),(3,2)),((4.2,2),(5,2)),((6.2,2),(7,2)),((8.2,2),(9.0,2))]:
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", lw=1.2))
# skip arc
ax.annotate("", xy=(9.0, 2), xytext=(1.7, 2),
            arrowprops=dict(arrowstyle="->", lw=1.5,
                             connectionstyle="arc3,rad=-0.4", color="red"))
ax.text(5, 3.4, "skip / residual (red): y = F(x) + x",
        ha="center", color="red", fontsize=11)
ax.text(9.6, 2, "y", fontsize=12, va="center")
plt.show()


## 4. Transfer learning — two stages

Training a 50-layer CNN from scratch needs millions of images. We have a few thousand. So we **transfer**:

1. Download ResNet50 weights pretrained on **ImageNet** (1.3M generic photos, 1000 classes). The early layers already know edges, textures, shapes — *generic* visual features.
2. Replace only the final classification head (1000-way → 6-way for our damage types).
3. **Stage 1** — freeze the backbone, train only the head for ~5 epochs. Fast — the head has very few weights.
4. **Stage 2** — unfreeze the top half of the backbone, train end-to-end at a *much lower* learning rate. Fine-tunes the high-level features to our domain (cars).

Why two stages? If you skip stage 1 and immediately fine-tune everything, the random head produces large gradient signals that smash the carefully-tuned backbone weights. Stage 1 lets the head learn first; stage 2 then nudges everything gently.


In [ ]:
# Look at the model the project actually trains.
from ccdp.models.damage_classifier import build_damage_classifier
m = build_damage_classifier(num_classes=6, pretrained=False)
print(type(m).__name__)
total = sum(p.numel() for p in m.parameters())
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,}")


## 5. Loss function for multi-label classification

For *single-label* (one class per image) we use **cross-entropy**. For *multi-label* (independent yes/no per class) we use **binary cross-entropy per class**:

$$
\mathcal{L} = -\sum_{c=1}^{C} \big[\,y_c \log \hat{y}_c + (1 - y_c) \log (1 - \hat{y}_c)\,\big]
$$

where $y_c \in \{0, 1\}$ is the true label for class $c$ and $\hat{y}_c \in (0, 1)$ is the sigmoid-activated logit.

The intuition: each class gets its own little binary classifier ("is this a dent? yes/no") and they all contribute independently to the total loss.


In [ ]:
# Demo BCE loss math by hand.
import numpy as np

# Ground truth: image has dent + scratch but no crack.
y_true = np.array([1, 1, 0])
# Model says: 0.9 dent, 0.4 scratch (too uncertain), 0.05 crack.
y_pred = np.array([0.9, 0.4, 0.05])

eps = 1e-9  # numerical guard
bce_per_class = -(y_true * np.log(y_pred + eps) + (1-y_true) * np.log(1-y_pred + eps))
print("BCE per class:", bce_per_class.round(4))
print("Total BCE   :", bce_per_class.sum().round(4))
print()
print("Notice scratch (0.4) dominates the loss — that's the class the model")
print("is least confident about, and it's the one the gradient will push hardest.")


## 6. Demo-scale training (runnable in ≤ 10 min on Colab)

We'll train for **just 2 epochs on a tiny synthetic subset** so you see the training loop end-to-end. This will NOT produce a competitive model — production weights come from the multi-hour run committed as v0.1.0.


In [ ]:
# Synthetic mini-batch — randomly generated tensors, not real data.
# This is enough to show the optimiser stepping and the loss going down.
import torch, torch.nn as nn
from ccdp.models.damage_classifier import build_damage_classifier
from ccdp.utils import pick_device

device = pick_device()
print("device:", device)

model = build_damage_classifier(num_classes=6, pretrained=False).to(device)
optim = torch.optim.AdamW(model.parameters(), lr=3e-4)
crit  = nn.BCEWithLogitsLoss()

# 16 fake images, 6 random binary labels each.
torch.manual_seed(0)
x = torch.randn(16, 3, 224, 224, device=device)
y = (torch.rand(16, 6, device=device) > 0.5).float()

for epoch in range(2):
    optim.zero_grad()
    logits = model(x)
    loss = crit(logits, y)
    loss.backward()
    optim.step()
    print(f"epoch {epoch+1}  loss={loss.item():.4f}")
print("\nThe loss came down — training loop works end-to-end.")


## 7. Optional — full training on the real dataset

Uncomment the cell below to launch the **production** training script. This is what produced the weights packaged in the GitHub Release. On Colab T4 it takes **~3 hours**; on a free CPU instance it will not finish in a session.

You will need to upload your Kaggle API token first — see the dataset-prep section of notebook 01.


In [ ]:
# Uncomment to run the real training. Keeps the demo notebook safe to "Run All".
#
# from ccdp.train.classifier import train_classifier
# train_classifier(
#     epochs_stage_1=5,
#     epochs_stage_2=15,
#     batch_size=32,
#     learning_rate_stage_1=1e-3,
#     learning_rate_stage_2=1e-4,
#     output_dir="checkpoints/classifier_full",
# )
print("(commented out — uncomment to run multi-hour training)")


**Next:** notebook 03 covers the **YOLOv8 detector** — how a model predicts boxes, what IoU and NMS mean, and how mAP is computed.
